In [ ]:
!pip install sae-lens transformer-lens


In [1]:
!pip install "numpy<2.0.0"

# Version 1  (Feature Extraction Patchscope) But cannot answer our question on actvating Neuron Y on Vector x

In [ ]:
import torch
import json
import gc
from datasets import load_dataset
from tqdm import tqdm
from transformer_lens import HookedTransformer
from sae_lens import SAE
from huggingface_hub import login 

HF_TOKEN = ""
login(token=HF_TOKEN)
print("Successfully logged into Hugging Face!")


gc.collect()
torch.cuda.empty_cache()

torch.set_grad_enabled(False)

# Check if we have multiple GPUs available
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPUs.")

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

target_layer = 12
json_file = f"gemma2_neuron_mapping_layer_{target_layer}.json"

print("\n--- Loading Gemma-2-2B and SAE ---")
model = HookedTransformer.from_pretrained(
    "gemma-2-2b", 
    device="cuda", 
    n_devices=n_gpus,              
    dtype=torch.bfloat16,
    center_unembed=False,          
    center_writing_weights=False   
)
layer_device = next(model.blocks[target_layer].parameters()).device
print(f"Target Layer {target_layer} successfully loaded onto {layer_device}")
# Load the SAE for Layer 12 (Residual Stream)
sae_id = f"layer_{target_layer}/width_16k/canonical"
sae = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id=sae_id,
    device=str(layer_device)
)
# PART 1: Build the Activation Atlas
print("\n--- PART 1: Profiling Dataset (Activation Atlas) ---")
dataset_profile = load_dataset("NeelNanda/pile-10k", split="train[100:250]")

activation_atlas = {}
hook_mlp_post = f"blocks.{target_layer}.mlp.hook_post"

print(f"Profiling layer {target_layer} neurons...")
for text_idx, example in enumerate(tqdm(dataset_profile)):
    text = example["text"]
    tokens = model.to_tokens(text)
    tokens = tokens[:, :128]
    
    _, cache = model.run_with_cache(tokens, names_filter=[hook_mlp_post])
    Y_acts = cache[hook_mlp_post][0] 
    
    seq_max_vals, seq_max_indices = Y_acts.max(dim=0)
    intermediate_size = Y_acts.shape[-1]
    
    for neuron_idx in range(intermediate_size):
        current_max = seq_max_vals[neuron_idx].item()
        
        if current_max > activation_atlas.get(neuron_idx, {}).get("activation_score", -1.0):
            token_pos = seq_max_indices[neuron_idx].item()
            trigger_token = model.tokenizer.decode([tokens[0, token_pos]])
            
            activation_atlas[neuron_idx] = {
                "text_index": text_idx,
                "token_position": token_pos,
                "trigger_token": trigger_token,
                "activation_score": current_max
            }

print(f"Exporting Activation Atlas to {json_file}...")
with open(json_file, "w") as f:
    json.dump(activation_atlas, f, indent=4)

# PART 2: Feature Extraction (Vector X)
print("\n--- PART 2: Feature Extraction ---")
best_neuron_idx = max(activation_atlas.keys(), key=lambda k: activation_atlas[k]["activation_score"])
best_neuron_data = activation_atlas[best_neuron_idx]

text_index = best_neuron_data["text_index"]
source_position = best_neuron_data["token_position"]
trigger_token = best_neuron_data["trigger_token"]

print(f"Targeting Neuron: {best_neuron_idx}")
print(f"Max Activation: {best_neuron_data['activation_score']:.4f} from token '{trigger_token}'")

dataset_target = load_dataset("NeelNanda/pile-10k", split=f"train[{text_index}:{text_index+1}]")
source_prompt = dataset_target[0]["text"]

source_tokens = model.to_tokens(source_prompt)
source_tokens = source_tokens[:, :128]

hook_resid_post = f"blocks.{target_layer}.hook_resid_post"

_, cache = model.run_with_cache(source_tokens, names_filter=[hook_resid_post])
extracted_X = cache[hook_resid_post][0, source_position, :].detach().clone()

print(f"Extracted Vector X from Layer {target_layer}, Position {source_position}")

# PART 3: SAE Verification
print("\n--- PART 3: SAE Verification ---")
feature_acts = sae.encode(extracted_X.unsqueeze(0).to(sae.device))
top_acts, top_indices = feature_acts[0].topk(5)

print("Top 5 Activated SAE Features in Vector X:")
for act, idx in zip(top_acts, top_indices):
    if act.item() > 0:
        print(f" - Feature_{idx.item()}: Activation = {act.item():.4f}")
        print(f"   Neuronpedia: https://neuronpedia.org/gemma-2-2b/{target_layer}-gemmascope-res-16k/{idx.item()}")

# PART 4: Patchscope Generation

print("\n--- PART 4: Patchscope Autoregressive Generation ---")
target_prompt = "A detailed description of x is that"
target_tokens = model.to_tokens(target_prompt)
target_str_tokens = model.to_str_tokens(target_prompt)


try:
    target_pos = next(i for i, t in enumerate(target_str_tokens) if 'x' in t.lower())
except StopIteration:
    target_pos = target_tokens.shape[1] - 3 

print(f"Target Prompt: '{target_prompt}'")
print(f"Injecting Vector X into token '{target_str_tokens[target_pos]}' at position {target_pos}")

def patch_x_hook_generation(resid, hook):
    # TransformerLens context manager bypasses autoregressive passes automatically 
    # if sequence length is 1 (during KV cached generation).
    if resid.shape[1] > target_pos:
        # Cast to match bfloat16
        resid[:, target_pos, :] = extracted_X.to(resid.dtype).to(resid.device)
    return resid

with model.hooks(fwd_hooks=[(hook_resid_post, patch_x_hook_generation)]):
    output_ids = model.generate(
        target_tokens,
        max_new_tokens=20,
        temperature=0.0,
        prepend_bos=False
    )

input_length = target_tokens.shape[1]
decoded_output = model.tokenizer.decode(output_ids[0, input_length:], skip_special_tokens=True)

print("\n=== Experiment Results ===")
print(f"Original Trigger Token: '{trigger_token}'")
print(f"Patched Target Prompt: '{target_prompt}'")
print(f"Generated Attributes (Post-Patch): '{decoded_output.strip()}'")

del model, sae, extracted_X, cache
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(f"Targeting Neuron: {best_neuron_idx}")
print(f"Original Max Activation: {best_neuron_data['activation_score']:.4f}")
print(f"Original Trigger Token: '{trigger_token}'")

# Version 2  does answer our question : 
The core differences between the two scripts lie in where the vector is extracted/injected and what is measured during the generation:

Extraction and Injection Point: * Version 1 uses hook_resid_post. This extracts and injects the vector after Layer 12 has completed all of its operations (Attention and MLP).

Version 2 uses hook_resid_pre. This extracts and injects the vector before Layer 12 processes the information.

Observation Mechanism: * Version 1 only looks at the final generated text to see what attributes the vector decodes into. It does not look inside the layer.

Version 2 introduces observe_mlp_hook at hook_mlp_post. This actively monitors and records the specific activation score of target neuron (best_neuron_idx) during the generation phase.

In [3]:
import torch
import json
import gc
from datasets import load_dataset
from tqdm import tqdm
from transformer_lens import HookedTransformer
from sae_lens import SAE
from huggingface_hub import login 

HF_TOKEN = ""
login(token=HF_TOKEN)
print("Successfully logged into Hugging Face!")


gc.collect()
torch.cuda.empty_cache()

torch.set_grad_enabled(False)

# Check if we have multiple GPUs available
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPUs.")

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

target_layer = 12
json_file = f"gemma2_neuron_mapping_layer_{target_layer}.json"

print("\n--- Loading Gemma-2-2B and SAE ---")

model = HookedTransformer.from_pretrained(
    "gemma-2-2b", 
    device="cuda", 
    n_devices=n_gpus,             
    dtype=torch.bfloat16,
    center_unembed=False,         
    center_writing_weights=False  
)
layer_device = next(model.blocks[target_layer].parameters()).device
print(f"Target Layer {target_layer} successfully loaded onto {layer_device}")
# Load the SAE for Layer 12 (Residual Stream)
sae_id = f"layer_{target_layer}/width_16k/canonical"
sae = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res-canonical",
    sae_id=sae_id,
    device=str(layer_device)
)


# PART 1: Build the Activation Atlas
print("\n--- PART 1: Profiling Dataset (Activation Atlas) ---")
dataset_profile = load_dataset("NeelNanda/pile-10k", split="train[100:250]")

activation_atlas = {}
hook_mlp_post = f"blocks.{target_layer}.mlp.hook_post"

print(f"Profiling layer {target_layer} neurons...")
for text_idx, example in enumerate(tqdm(dataset_profile)):
    text = example["text"]
    tokens = model.to_tokens(text)
    tokens = tokens[:, :128]
    
    _, cache = model.run_with_cache(tokens, names_filter=[hook_mlp_post])
    Y_acts = cache[hook_mlp_post][0] 
    
    seq_max_vals, seq_max_indices = Y_acts.max(dim=0)
    intermediate_size = Y_acts.shape[-1]
    
    for neuron_idx in range(intermediate_size):
        current_max = seq_max_vals[neuron_idx].item()
        
        if current_max > activation_atlas.get(neuron_idx, {}).get("activation_score", -1.0):
            token_pos = seq_max_indices[neuron_idx].item()
            trigger_token = model.tokenizer.decode([tokens[0, token_pos]])
            
            activation_atlas[neuron_idx] = {
                "text_index": text_idx,
                "token_position": token_pos,
                "trigger_token": trigger_token,
                "activation_score": current_max
            }

print(f"Exporting Activation Atlas to {json_file}...")
with open(json_file, "w") as f:
    json.dump(activation_atlas, f, indent=4)

# PART 2: Feature Extraction (Vector X)
print("\n--- PART 2: Feature Extraction ---")
best_neuron_idx = max(activation_atlas.keys(), key=lambda k: activation_atlas[k]["activation_score"])
best_neuron_data = activation_atlas[best_neuron_idx]

text_index = best_neuron_data["text_index"]
source_position = best_neuron_data["token_position"]
trigger_token = best_neuron_data["trigger_token"]

print(f"Targeting Neuron: {best_neuron_idx}")
print(f"Max Activation: {best_neuron_data['activation_score']:.4f} from token '{trigger_token}'")

dataset_target = load_dataset("NeelNanda/pile-10k", split=f"train[{text_index}:{text_index+1}]")
source_prompt = dataset_target[0]["text"]
source_tokens = model.to_tokens(source_prompt)
source_tokens = source_tokens[:, :128]

# Grab 5 tokens before and 5 tokens after the trigger token to see the context
window_start = max(0, source_position - 5)
window_end = min(source_tokens.shape[1], source_position + 6)
context_tokens = source_tokens[0, window_start:window_end]
trigger_context = model.tokenizer.decode(context_tokens)

# CHANGE 1: Use hook_resid_pre to get the vector before Layer 12 processes it
hook_resid_pre = f"blocks.{target_layer}.hook_resid_pre"

_, cache = model.run_with_cache(source_tokens, names_filter=[hook_resid_pre])
extracted_X = cache[hook_resid_pre][0, source_position, :].detach().clone()

print(f"Extracted Vector X from Layer {target_layer}, Position {source_position}")

# PART 3: SAE Verification
print("\n--- PART 3: SAE Verification ---")
feature_acts = sae.encode(extracted_X.unsqueeze(0).to(sae.device))
top_acts, top_indices = feature_acts[0].topk(5)

print("Top 5 Activated SAE Features in Vector X:")
for act, idx in zip(top_acts, top_indices):
    if act.item() > 0:
        print(f" - Feature_{idx.item()}: Activation = {act.item():.4f}")
        print(f"   Neuronpedia: https://neuronpedia.org/gemma-2-2b/{target_layer}-gemmascope-res-16k/{idx.item()}")

# PART 4: Patchscope Generation
print("\n--- PART 4: Patchscope Autoregressive Generation ---")
target_prompt = "A detailed description of x is that"
target_tokens = model.to_tokens(target_prompt)
target_str_tokens = model.to_str_tokens(target_prompt)


try:
    target_pos = next(i for i, t in enumerate(target_str_tokens) if 'x' in t.lower())
except StopIteration:
    target_pos = target_tokens.shape[1] - 3 

print(f"Target Prompt: '{target_prompt}'")
print(f"Injecting Vector X into token '{target_str_tokens[target_pos]}' at position {target_pos}")

observed_activation = 0.0


def observe_mlp_hook(mlp_post, hook):
    global observed_activation
    if mlp_post.shape[1] > target_pos:
        # Save the activation score of our specific neuron at the target position
        observed_activation = mlp_post[0, target_pos, best_neuron_idx].item()
    return mlp_post

# CHANGE 3: Update the patch hook to use your new injection point
def patch_x_hook_generation(resid, hook):
    if resid.shape[1] > target_pos:
        resid[:, target_pos, :] = extracted_X.to(resid.dtype).to(resid.device)
    return resid

# CHANGE 4: Register BOTH hooks (one to inject, one to observe)
with model.hooks(fwd_hooks=[
    (hook_resid_pre, patch_x_hook_generation),  # Inject before Layer 12
    (hook_mlp_post, observe_mlp_hook)           # Observe inside Layer 12
]):
    output_ids = model.generate(
        target_tokens,
        max_new_tokens=20,
        temperature=0.0,
        prepend_bos=False
    )
input_length = target_tokens.shape[1]
decoded_output = model.tokenizer.decode(output_ids[0, input_length:], skip_special_tokens=True)

print("\n=== Experiment Results ===")
print("\n=== Experiment Results ===")
print(f"Original Source Prompt: '{source_prompt}'")   
print(f"Original Trigger Token: '{trigger_token}'")
print(f"Patched Target Prompt: '{target_prompt}'")
print(f"Generated Attributes (Post-Patch): '{decoded_output.strip()}'")

print(f"Targeting Neuron: {best_neuron_idx}")
print(f"Original Max Activation: {best_neuron_data['activation_score']:.4f}")
print(f"Observed Activation Post-Patch: {observed_activation:.4f}")
print(f"Trigger Context (±5 tokens): '...{trigger_context}...'")

print(f"\nNeuron {best_neuron_idx} activation during patchscope: {observed_activation:.4f}")

del model, sae, extracted_X, cache
gc.collect()
torch.cuda.empty_cache()

Successfully logged into Hugging Face!
Detected 2 GPUs.
Using device: cuda

--- Loading Gemma-2-2B and SAE ---


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Target Layer 12 successfully loaded onto cuda:0

--- PART 1: Profiling Dataset (Activation Atlas) ---
Profiling layer 12 neurons...


100%|██████████| 150/150 [01:05<00:00,  2.31it/s]


Exporting Activation Atlas to gemma2_neuron_mapping_layer_12.json...

--- PART 2: Feature Extraction ---
Targeting Neuron: 8526
Max Activation: 9.3750 from token '3'
Extracted Vector X from Layer 12, Position 94

--- PART 3: SAE Verification ---
Top 5 Activated SAE Features in Vector X:
 - Feature_315: Activation = 41.0987
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/12-gemmascope-res-16k/315
 - Feature_6810: Activation = 31.1544
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/12-gemmascope-res-16k/6810
 - Feature_2291: Activation = 24.8089
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/12-gemmascope-res-16k/2291
 - Feature_164: Activation = 18.4591
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/12-gemmascope-res-16k/164
 - Feature_11195: Activation = 17.2775
   Neuronpedia: https://neuronpedia.org/gemma-2-2b/12-gemmascope-res-16k/11195

--- PART 4: Patchscope Autoregressive Generation ---
Target Prompt: 'A detailed description of x is that'
Injecting Vector X into tok

  0%|          | 0/20 [00:00<?, ?it/s]


=== Experiment Results ===

=== Experiment Results ===
Original Source Prompt: 'Udzungwa red colobus

The Uzungwa red colobus (Piliocolobus gordonorum), also known as the Udzungwa red colobus or Iringa red colobus, is a species of primate in the family Cercopithecidae. It is endemic to riverine and montane forest in the Udzungwa Mountains in Tanzania. It is threatened by habitat loss.

References

Uzungwa red colobus
Category:Endemic fauna of Tanzania
Category:Mammals of Tanzania
Category:Endangered fauna of Africa
Uzungwa red colobus
Category:Taxonomy articles created by Polbot
Category:Primates of Africa'
Original Trigger Token: '3'
Patched Target Prompt: 'A detailed description of x is that'
Generated Attributes (Post-Patch): 'it is a set of all the possible outcomes of an experiment.

The set of all possible outcomes'
Targeting Neuron: 8526
Original Max Activation: 9.3750
Observed Activation Post-Patch: 3.1406
Trigger Context (±5 tokens): '...wa red colobus
Category:Endemic fauna 